# ⬡ Dabba 8B — Fine-tune Llama 3.1 8B into your personal AI (Kaggle version)

**Before running, check the right panel:**
1. `Settings → Accelerator → GPU T4 x2` (or P100)
2. `Settings → Internet → On` (required for pip installs + downloading the base model/datasets)

## ⚠️ Critical: how you run this determines whether a crash loses your progress

- **Clicking cells one by one (interactive)** → if the tab closes, the connection drops, or Kaggle
  reclaims the session, **everything in `/kaggle/working/` — including your checkpoints — is lost.**
  This is almost certainly why a previous run "hung and stopped after 5 hours" with nothing to resume from.
- **Save Version → Save & Run All (Commit)** → runs the whole notebook server-side. Your browser can
  close, your laptop can sleep, none of it matters. When it finishes (or even if it fails partway),
  the **Output tab** on that version has everything written to `/kaggle/working/` up to that point,
  including every checkpoint. This is the only way to reliably keep multi-hour progress on Kaggle.

**Always use Commit for a run you don't want to lose.** Interactive mode is fine only for quickly
testing that cells run without errors on a small slice of data.

What this does:
- Takes Llama 3.1 8B (already smart)
- Fine-tunes it with YOUR Dabba personality + coding skills
- Exports a `.gguf` file you download and run locally forever (no API key)

**Time on Kaggle T4:** ~2-3 hours

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())
# Should show something like: Tesla T4, 15360 MiB (x2 on Kaggle)
# If it errors or says nothing → Settings panel (right side) → Accelerator → GPU T4 x2

In [ ]:
# ── Cell 2: Install (run once, ~3 min) ────────────────────────────────────────
%%capture
!pip install unsloth
!pip install --no-deps "trl>=0.9.0" peft accelerate bitsandbytes datasets
print('done')

In [ ]:
# ── Cell 3: Verify install ─────────────────────────────────────────────────────
import unsloth, trl, peft, transformers
print(f'unsloth {unsloth.__version__}  trl {trl.__version__}  peft {peft.__version__}  transformers {transformers.__version__}')
print('✓ All packages ready')

In [ ]:
# ── Cell 4: Load Llama 3.1 8B in 4-bit ────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,      # auto-detect
    load_in_4bit   = True,      # saves ~10GB VRAM
)

used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✓ Model loaded  —  VRAM used: {used:.1f}GB / {total:.1f}GB')

In [ ]:
# ── Cell 5: Add LoRA adapters ──────────────────────────────────────────────────
# r=16 is safe on a single T4 (r=64 would OOM). Still gives great results.
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # saves 30% VRAM
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'✓ LoRA ready — trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)')

In [ ]:
# ── Cell 6: Build dataset ──────────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets

# YOUR Dabba personality — edit this to make it yours
DABBA_SYSTEM = (
    "You are Dabba, a highly capable personal AI assistant. "
    "You are direct, concise, and technically excellent. "
    "You help with coding, analysis, writing, and reasoning. "
    "When writing code, you always write clean working code with brief comments on non-obvious logic. "
    "You are honest about what you know and don't know."
)

def fmt_alpaca(ex):
    instruction = (ex.get('instruction') or '').strip()
    inp         = (ex.get('input') or '').strip()
    output      = (ex.get('output') or '').strip()
    if not instruction or not output:
        return {'text': ''}
    user = f"{instruction}\n\n{inp}" if inp else instruction
    msgs = [
        {"role": "system",    "content": DABBA_SYSTEM},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": output},
    ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

def fmt_hermes(ex):
    convs = ex.get('conversations', [])
    if len(convs) < 2:
        return {'text': ''}
    msgs = [{"role": "system", "content": DABBA_SYSTEM}]
    for c in convs:
        role = "user" if c.get('from') == 'human' else "assistant"
        msgs.append({"role": role, "content": (c.get('value') or '').strip()})
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

print('Downloading datasets...')

# General assistant — high quality conversations
hermes_raw = load_dataset("teknium/OpenHermes-2.5", split="train[:10000]")
hermes     = hermes_raw.map(fmt_hermes, remove_columns=hermes_raw.column_names)

# Coding — code generation + explanation
code_raw = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
code     = code_raw.map(fmt_alpaca, remove_columns=code_raw.column_names)

# Instruction following — general tasks
alpaca_raw = load_dataset("tatsu-lab/alpaca", split="train[:5000]")
alpaca     = alpaca_raw.map(fmt_alpaca, remove_columns=alpaca_raw.column_names)

# Merge, clean, shuffle
combined = concatenate_datasets([hermes, code, alpaca])
combined = combined.filter(lambda x: len(x['text'].strip()) > 80)
combined = combined.shuffle(seed=42)

print(f'✓ Dataset: {len(combined):,} examples')
print('Sample preview:')
print(combined[0]['text'][:300])

In [ ]:
# ── Cell 7: Train (auto-resumes from the latest checkpoint if one exists) ──────
# Checkpoints land in /kaggle/working/dabba-checkpoints. This directory only
# survives a crash if you're running via Save & Run All (Commit) — clicking
# cells interactively and then losing the tab loses this directory too, since
# Kaggle only preserves /kaggle/working/ for committed runs, not live sessions.
import os
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

CHECKPOINT_DIR = "dabba-checkpoints"
FULL_RUN_STEPS = 4377

# Auto-detect the latest checkpoint instead of typing it in by hand each time.
latest_checkpoint = None
if os.path.exists(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        latest_checkpoint = os.path.join(
            CHECKPOINT_DIR, max(checkpoints, key=lambda d: int(d.split("-")[-1]))
        )

if latest_checkpoint:
    print(f'Found existing checkpoint: {latest_checkpoint} — resuming from there.')
else:
    print('No checkpoint found — starting a fresh run.')

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = combined,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing        = True,   # pack short examples together — 2x faster
    args = SFTConfig(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,   # effective batch = 8
        warmup_steps                 = 50,
        max_steps                    = FULL_RUN_STEPS,
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,           # more frequent progress updates
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        lr_scheduler_type            = "cosine",
        seed                         = 42,
        output_dir                   = CHECKPOINT_DIR,
        save_strategy                = "steps",
        save_steps                   = 50,            # save more often — smaller loss window on a crash
        save_total_limit             = 5,             # keep only the last 5 checkpoints (disk space)
        report_to                    = "none",        # disable wandb
    ),
)

used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM before training: {used:.1f}GB')
print('🚀 Training started...')

stats = trainer.train(resume_from_checkpoint=latest_checkpoint)

print(f'\n✓ Training complete!')
print(f'  Runtime : {stats.metrics["train_runtime"]/60:.1f} min')
print(f'  Final loss: {stats.metrics["train_loss"]:.4f}')

In [ ]:
# ── Cell 8: Quick test — does it respond like Dabba? ──────────────────────────
FastLanguageModel.for_inference(model)

def ask_dabba(question):
    msgs = [
        {"role": "system", "content": DABBA_SYSTEM},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        input_ids=inputs, max_new_tokens=400,
        temperature=0.7, do_sample=True,
    )
    reply = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'You: {question}')
    print(f'Dabba: {reply}\n')

ask_dabba("Who are you?")
ask_dabba("Write a Python function to check if a string is a palindrome.")

In [ ]:
# ── Cell 9: Export to GGUF ─────────────────────────────────────────────────────
# Q4_K_M = best quality/size balance (~4.7GB file).
# Saved into /kaggle/working/ (this notebook's CWD) so it appears in the
# "Output" tab automatically — no explicit save/mount step needed, unlike Colab.
print('Exporting to GGUF... (takes ~5-10 min)')

model.save_pretrained_gguf(
    "dabba-8b-gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

import os
files = os.listdir('dabba-8b-gguf')
for f in files:
    size = os.path.getsize(f'dabba-8b-gguf/{f}') / 1e9
    print(f'  {f}  ({size:.2f} GB)')
print('✓ GGUF export done — find it in the "Output" tab on the right once this notebook finishes/commits')

## ✅ Download your model from Kaggle

Unlike Colab, there's **no Drive-mount step** — Kaggle automatically keeps anything you write to
`/kaggle/working/` (this notebook's current directory) as the notebook's **Output**, but only for
**committed** versions (see the warning at the top — interactive sessions don't persist this).

### To get the file:
1. Click **Save Version → Save & Run All (Commit)** (top right) — this re-runs the whole notebook
   in the background on Kaggle's servers, even if you close the tab
2. Once it finishes, go to the notebook's **Output** tab
3. Download `dabba-8b-gguf/dabba-8b-Q4_K_M.gguf` (~4.7GB) directly to your PC

---

## After downloading — run Dabba locally

```bash
export DABBA_MODEL_PATH="/home/hasheem/dabba-8b-Q4_K_M.gguf"
cd "/home/hasheem/Hasheem files/Hasheem sub foders/ai"
python3 -m dabba.api.server
```

No Ollama, no Modelfile needed — Dabba loads the `.gguf` directly (requires `llama-cpp-python`
installed first: `sudo apt-get install -y ninja-build cmake && pip install llama-cpp-python`).

---

### Resuming after a Kaggle session limit / interruption

Cell 7 now **auto-detects the latest checkpoint** in `dabba-checkpoints/` — you don't need to type
a checkpoint name in by hand anymore. Checkpoints save every 50 steps (instead of 200), so a crash
loses at most a few minutes of progress instead of a big chunk.

1. Go to the notebook's **Output** tab from your last committed version
2. **Add Input → Notebook Output Files → this notebook** (this makes the last run's files available)
3. Copy `dabba-checkpoints/` from that input back into `/kaggle/working/`
4. Re-run Cells 1–6, then Cell 7 — it will print `Found existing checkpoint: ... — resuming from there`
   and continue automatically
5. Continue with Cells 8–9 as normal
6. **Commit again** — don't just run interactively, or you'll lose this recovered progress too